In [5]:
import re
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import torch.utils.data as data

from collections import Counter
import json

In [6]:
# load data from json
with open('train.json', 'r') as train_file:
    train_data = json.load(train_file)

print("Train data loaded:", len(train_data), "samples")
print(f"{train_data[0]}\n{train_data[1]}")

Train data loaded: 7401 samples
{'reviews': 'I bought this belt for my daughter in-law for Christmas and she loved it', 'sentiments': 1}
{'reviews': 'The size was perfect and so was the color.  It looked just like it did on the web page:)', 'sentiments': 1}


In [7]:
def tokenize(text):
    text = text.lower()
    text = text.replace("’", "'") # replace fancy apostrophes with normal ones
    text = text.replace("‘", "'") # replace fancy opening apostrophes with normal ones
    text = text.replace("n't", "nt") # remove apostrophes from can't, won't, and keep it's, we'll, ect
    text = text.replace(r"(http)\S+", "") # remove URLs
    text = re.sub(r"([a-z])\1{2,}", r"\1", text) # soooo -> so
    return re.findall(r"[a-z0-9]+|[!?]", text) # remove punctuation except for ! and ?

MAX_VOCAB = 20000
counter = Counter()
for item in train_data:
    counter.update(tokenize(item['reviews']))
most_common = counter.most_common(MAX_VOCAB - 2)

vocab = {"<pad>": 0, "<unk>": 1}
for word, _ in most_common:
    vocab[word] = len(vocab)

vocab_size = min(len(vocab), MAX_VOCAB)
print("Vocabulary size:", vocab_size)

Vocabulary size: 14082


In [8]:
print(re.sub(r"([a-z])\1{2,}", r"\1", "sooo"))

so


In [9]:
def encode(text):
    tokens = tokenize(text)
    return [vocab.get(t, 1) for t in tokens]  

encoded = [encode(item['reviews']) for item in train_data]
import numpy as np
lens = np.array([len(s) for s in encoded])
print("median:", np.median(lens), "90th pct:", np.percentile(lens, 90), "95th pct:", np.percentile(lens, 95))

median: 43.0 90th pct: 111.0 95th pct: 148.0


In [10]:
MAX_LEN = 120
padded = [(ids + [0] * MAX_LEN)[:MAX_LEN] for ids in encoded]

X = torch.tensor(padded)
y_true = torch.tensor([item['sentiments'] for item in train_data], dtype=torch.float32)
print("X shape:", X.shape)
print("y_true shape:", y_true.shape)

X shape: torch.Size([7401, 120])
y_true shape: torch.Size([7401])


In [11]:
dataset = data.TensorDataset(X, y_true)   # keeps each X row paired with its label

n_total = len(dataset)
n_val = int(0.15 * n_total)
n_train = n_total - n_val

generator = torch.Generator().manual_seed(42)   # reproducible split
train_set, val_set = data.random_split(dataset, [n_train, n_val], generator=generator)

train_loader = data.DataLoader(train_set, batch_size=64, shuffle=True)
val_loader = data.DataLoader(val_set, batch_size=64)

In [12]:
class Model(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=128):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embedding_dim, 
            hidden_dim, 
            num_layers=1, 
            batch_first=True, 
            bidirectional=True, 
            dropout=0)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_dim * 2, 1) # 1 for binary classification
    
    def forward(self, inputs):
        ## Embedding layer (+ Dropout) ##
        embeds = self.dropout(self.embeddings(inputs))

        ## Bidirectional LSTM layer ##
        lengths = (inputs != 0).sum(dim=1).clamp(min=1).cpu() 
        # Pack the padded sequence for LSTM processing, and unpack it after
        packed = pack_padded_sequence(
            embeds, lengths, batch_first=True, enforce_sorted=False)
        packed_out, _ = self.lstm(packed)
        padded, _ = pad_packed_sequence(
            packed_out, batch_first=True, padding_value=-1e9, total_length=inputs.size(1))

        # Max pooling over the time dimension (dim=1) to get a fixed-size representation
        pooled = padded.max(dim=1).values 

        ## Linear layer (+ Dropout) ##
        return self.fc(self.dropout(pooled)).squeeze(1)

model = Model(vocab_size)
print("Model Arch")
print(model)

Model Arch
Model(
  (embeddings): Embedding(14082, 128, padding_idx=0)
  (lstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=256, out_features=1, bias=True)
)


In [13]:
# Hyperparameters
learning_rate = 1e-2
weight_decay = 1e-2
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
loss_fn = nn.BCEWithLogitsLoss() # Binary Cross Entropy With Logits Loss

In [14]:
def run_epoch(loader, train=True):
    if train:
        model.train()      # dropout ON
    else:
        model.eval()       # dropout OFF

    total_loss = 0.0       # running sum of loss over all reviews
    correct = 0            # how many predictions were right
    count = 0              # how many reviews we've seen

    with torch.set_grad_enabled(train):    
        for xb, yb in loader:
            ### FORWARD PASS ###
            y_hat = model(xb)                  

            ### CALCULATE LOSS ###
            loss = loss_fn(y_hat, yb)

            ### BACKWARD PASS ###
            if train:
                optimizer.zero_grad()
                loss.backward()
                # nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            ### BOOKKEEPING ###
            total_loss += loss.item() * xb.size(0)
            preds = (torch.sigmoid(y_hat) > 0.5).float()
            correct += (preds == yb).sum().item()
            count += xb.size(0)

    return total_loss / count, correct / count     # average loss, accuracy

In [15]:
epochs = 10
best_val_loss = float("inf")

for epoch in range(1, epochs+1):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)

    print(f"Epoch {epoch}: train loss={train_loss:.4f} acc={train_acc:.3f} | "
          f"val loss={val_loss:.4f} acc={val_acc:.3f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            "model_state": model.state_dict(),
            "epoch": epoch,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "train_loss": train_loss,
            "train_acc": train_acc,
        }, "best_model.pt")

Epoch 1: train loss=0.3876 acc=0.852 | val loss=0.2223 acc=0.904
Epoch 2: train loss=0.2566 acc=0.891 | val loss=0.1812 acc=0.932
Epoch 3: train loss=0.1901 acc=0.927 | val loss=0.1979 acc=0.922
Epoch 4: train loss=0.1589 acc=0.940 | val loss=0.2258 acc=0.932
Epoch 5: train loss=0.1226 acc=0.956 | val loss=0.2178 acc=0.933
Epoch 6: train loss=0.0920 acc=0.967 | val loss=0.1999 acc=0.941
Epoch 7: train loss=0.0852 acc=0.969 | val loss=0.1874 acc=0.935
Epoch 8: train loss=0.0594 acc=0.979 | val loss=0.2407 acc=0.926
Epoch 9: train loss=0.0551 acc=0.980 | val loss=0.2088 acc=0.939
Epoch 10: train loss=0.0542 acc=0.981 | val loss=0.2160 acc=0.935


In [16]:
checkpoint = torch.load("best_model.pt")

model = Model(vocab_size)                          
model.load_state_dict(checkpoint["model_state"])
model.eval()                                       

print("Best epoch:", checkpoint["epoch"])
print("Val loss:", checkpoint["val_loss"])
print("Val acc:", checkpoint["val_acc"])

Best epoch: 2
Val loss: 0.18117140536909704
Val acc: 0.9324324324324325
